# Lab 4, table maintenance

slv_sales_order_lines is a good real test case for this one without me needing to fake anything. The schema evolution notebook left behind a few small 5 row appends in it from testing mergeSchema and widening, on top of the original bulk load. That's exactly the kind of fragmented file layout OPTIMIZE is meant to clean up, and I caused it myself honestly, just as a side effect of earlier testing.%md


In [0]:
dbutils.widgets.text("catalog", "lab4", "Catalog")
catalog = dbutils.widgets.get("catalog")

## Checking the file layout before doing anything

Given the bulk load plus a handful of tiny test writes, I'd expect more files than the actual data volume really needs.

In [0]:
%sql
DESCRIBE DETAIL lab4.silver.slv_sales_order_lines

## OPTIMIZE

In [0]:
%sql
OPTIMIZE lab4.silver.slv_sales_order_lines

In [0]:
%sql
-- comparing against the earlier DESCRIBE DETAIL, numFiles should be lower now
DESCRIBE DETAIL lab4.silver.slv_sales_order_lines

## Liquid clustering vs zorder vs classic partitioning

For my tables here, around 8000 rows, none of this actually changes anything measurable, the point of this section is really just understanding the trade offs, since it matters a lot more once a table is millions or billions of rows. I'm using liquid clustering below since it's the current recommendation and needs the least ongoing decision making, not because this specific table is anywhere near large enough to need it.

In [0]:
%sql
-- customer_id and order_datetime are the two columns I'd most likely filter on in real queries
ALTER TABLE lab4.silver.slv_sales_order_lines
CLUSTER BY (customer_id, order_datetime)

In [0]:
%sql
-- clustering only applies to future OPTIMIZE runs, not retroactively just by declaring CLUSTER BY,
-- so running OPTIMIZE once here to actually cluster the existing data now
OPTIMIZE lab4.silver.slv_sales_order_lines

In [0]:
%sql
DESCRIBE DETAIL lab4.silver.slv_sales_order_lines

For reference, this is roughly what the older zorder syntax would have looked like before liquid clustering existed. Not actually running this since I already committed to CLUSTER BY on this table above, and mixing zorder and cluster by on the same table isn't supported anyway:

```sql
OPTIMIZE lab4.silver.slv_sales_order_lines
ZORDER BY (customer_id, order_datetime)
```

## VACUUM

Running DRY RUN first.

In [0]:
%sql
VACUUM lab4.silver.slv_sales_order_lines DRY RUN

Since everything in this table was written today, the default 7 day window means this should come back empty, nothing eligible yet. That's expected, not a bug, VACUUM is meant to be conservative by default so it can't accidentally break time travel or a read that's still in progress.

Worth knowing that the retention window can be shortened, VACUUM RETAIN 0 HOURS with the retention check disabled, but I'm not doing that here on purpose. Doing that for real risks deleting a file some other reader or a time travel query still needs, not something to do casually just to make a demo show a bigger number.

In [0]:
%sql
VACUUM lab4.silver.slv_sales_order_lines

## Same maintenance on the customers table

Different clustering key here, just customer_id plus effective_start makes more sense for this one.

I originally tried clustering on customer_id and is_current together, but Delta rejected it, is_current is a boolean and liquid clustering doesn't support boolean columns as clustering keys, makes sense actually, a column with only two possible values doesn't really give Delta anything to skip on. Swapped it for effective_start instead.

In [0]:
%sql
ALTER TABLE lab4.silver.slv_customers_scd2
CLUSTER BY (customer_id, effective_start)

In [0]:
%sql
OPTIMIZE lab4.silver.slv_customers_scd2

In [0]:
%sql
VACUUM lab4.silver.slv_customers_scd2 DRY RUN